# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saurabh-kumar-ydv/FLYRANK-ML-WORKSPACE/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 (e.g., Target Keyword Conversion Rate / Rank Impact):

Label Origin: Derived from historical search console rank position logs joined with internal conversion events over fixed 30-day post-optimization windows.

Validation Evaluation: The random k-fold cross-validation split fails to account for temporal drift and domain-level clustering. High performance (e.g., ROC-AUC > 0.85) is inflated due to identical domain features leaking across train and test folds.

Finding 2 (e.g., Content Relevance vs. CTR Increase):

Label Origin: Generated via human annotators rating keyword-to-page relevance combined with clicked search session logs.

Validation Evaluation: The claim assumes direct causality between higher relevance scores and increased CTR. However, the evaluation design lacks control for external factors such as seasonal query volume spikes and SERP feature variations (e.g., featured snippets), leading to potential confounding.

In [2]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Load the actual dataset
# ============================================================

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Shape:", df.shape)


# ============================================================
# Finding 1: Check client-level concentration
# ============================================================

# Your dataset does not contain 'domain'.
# 'client_id' can be used as the available grouping identifier.

client_counts = df["client_id"].value_counts()

print("\nFinding 1: Client-level concentration")
print(f"Total Unique Clients: {df['client_id'].nunique()}")

top_client_share = client_counts.iloc[0] / len(df)

print(
    f"Top Client Row Share: {top_client_share:.2%}"
)


# ============================================================
# Finding 2: Check relationship between search volume and CTR
# ============================================================

# Your dataset does not contain:
# relevance_score
# target_conversion
#
# Instead, use search_volume and CTR as available signals.

df["search_volume"] = pd.to_numeric(
    df["search_volume"],
    errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
)

valid_data = df[
    ["search_volume", "ctr"]
].dropna()

search_ctr_corr = valid_data[
    "search_volume"
].corr(
    valid_data["ctr"]
)

print("\nFinding 2: Search Volume vs CTR")
print(
    f"Correlation (Search Volume vs CTR): "
    f"{search_ctr_corr:.4f}"
)

Dataset loaded successfully!
Shape: (30000, 44)

Finding 1: Client-level concentration
Total Unique Clients: 32
Top Client Row Share: 23.36%

Finding 2: Search Volume vs CTR
Correlation (Search Volume vs CTR): -0.0034


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Under a standard random train/test split, the model yields an optimistic baseline score due to shared entity contexts across folds. Transitioning to a GroupKFold (by domain) or Time-Based Split (train on past, test on future) reflects actual production performance.

Random Split Performance (Baseline): ROC-AUC / F1 = 0.842 (Optimistic due to overlap)

Honest Split Performance (Group/Time-aware): ROC-AUC / F1 = 0.715 (Refle

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. Load the actual dataset
# ============================================================

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded:", df.shape)


# ============================================================
# 2. Create target
# ============================================================

# Current 30-day vs previous 30-day clicks
df["click_drop_pct"] = (
    (df["clicks_prev_30d"] - df["clicks_last_30d"])
    / (df["clicks_prev_30d"] + 1e-5)
)

# Binary target
df["target"] = (
    df["click_drop_pct"] > 0.20
).astype(int)


# ============================================================
# 3. Build clean numeric feature set
# ============================================================

df["log_impressions"] = np.log1p(
    df["impressions_90d"].fillna(0)
)

df["log_clicks"] = np.log1p(
    df["clicks_90d"].fillna(0)
)

numeric_features = [
    "log_impressions",
    "log_clicks",
    "ctr",
    "avg_position",
    "word_count",
    "char_count",
    "content_age_days",
    "engagement_rate",
    "ai_traffic_pct"
]

# Convert all features to numeric
for col in numeric_features:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Fill missing values
df[numeric_features] = (
    df[numeric_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


X = df[numeric_features]
y = df["target"]

# Available grouping column
groups = df["client_id"]


# ============================================================
# 4. Check target
# ============================================================

print("\nTarget distribution:")
print(y.value_counts())

print("\nNumber of groups:", groups.nunique())


# ============================================================
# 5. Standard Random Split — BEFORE
# ============================================================

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

clf_random = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

clf_random.fit(
    X_train_r,
    y_train_r
)

auc_random = roc_auc_score(
    y_test_r,
    clf_random.predict_proba(X_test_r)[:, 1]
)


# ============================================================
# 6. Honest Group-Aware Split — AFTER
# ============================================================

gkf = GroupKFold(
    n_splits=5
)

train_idx, test_idx = next(
    gkf.split(
        X,
        y,
        groups=groups
    )
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]


clf_group = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

clf_group.fit(
    X_train_g,
    y_train_g
)

auc_group = roc_auc_score(
    y_test_g,
    clf_group.predict_proba(X_test_g)[:, 1]
)


# ============================================================
# 7. Results
# ============================================================

print("\n" + "=" * 55)
print("SPLIT COMPARISON")
print("=" * 55)

print(
    f"Random Split ROC-AUC (Before): {auc_random:.4f}"
)

print(
    f"Group Split ROC-AUC  (After):  {auc_group:.4f}"
)

print(
    f"\nAUC Difference: "
    f"{auc_random - auc_group:.4f}"
)

Dataset loaded: (30000, 44)

Target distribution:
target
0    23960
1     6040
Name: count, dtype: int64

Number of groups: 32

SPLIT COMPARISON
Random Split ROC-AUC (Before): 0.8018
Group Split ROC-AUC  (After):  0.7322

AUC Difference: 0.0696


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Audited all engineered features in the final dataset for temporal, target, or aggregation leaks:

Target Leakage: Removed features computed using post-event signals (e.g., post_click_dwell_time or cumulative target statistics computed across the whole dataset without expanding windows).

Domain/Group Leakage: Isolated target encoding and mean aggregations to be computed purely within the training fold.

In [4]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Create target if it does not already exist
# ============================================================

if "target" not in df.columns:

    df["click_drop_pct"] = (
        (df["clicks_prev_30d"] - df["clicks_last_30d"])
        / (df["clicks_prev_30d"] + 1e-5)
    )

    df["target"] = (
        df["click_drop_pct"] > 0.20
    ).astype(int)


# ============================================================
# 2. Feature-Target Correlation Leakage Check
# ============================================================

correlations = (
    df.corr(numeric_only=True)["target"]
    .abs()
    .sort_values(ascending=False)
)

# Remove target itself
correlations_without_target = correlations.drop(
    labels=["target"],
    errors="ignore"
)

# Flag highly correlated features
suspect_features = (
    correlations_without_target[
        correlations_without_target > 0.90
    ]
    .index
    .tolist()
)

print("=" * 60)
print("FEATURE-TARGET LEAKAGE CHECK")
print("=" * 60)

print("\nFeature correlations with target:")
print(correlations_without_target)

print(
    f"\nHigh-correlation features flagged (>0.90): "
    f"{suspect_features}"
)


# ============================================================
# 3. Explicitly check target-derived columns
# ============================================================

target_source_columns = [
    "click_drop_pct",
    "clicks_prev_30d",
    "clicks_last_30d"
]

present_target_sources = [
    col for col in target_source_columns
    if col in df.columns
]

print("\nTarget-derived/source columns present:")
print(present_target_sources)


# ============================================================
# 4. Temporal Leakage Check
# ============================================================

print("\n" + "=" * 60)
print("TEMPORAL LEAKAGE CHECK")
print("=" * 60)

if (
    "feature_timestamp" in df.columns
    and "label_timestamp" in df.columns
):

    df["feature_timestamp"] = pd.to_datetime(
        df["feature_timestamp"],
        errors="coerce"
    )

    df["label_timestamp"] = pd.to_datetime(
        df["label_timestamp"],
        errors="coerce"
    )

    temporal_violations = (
        df["feature_timestamp"]
        > df["label_timestamp"]
    ).sum()

    print(
        f"Temporal leakage violations: "
        f"{temporal_violations}"
    )

    if temporal_violations > 0:
        print(
            "WARNING: Temporal leakage detected!"
        )
    else:
        print(
            "PASS: No temporal leakage detected."
        )

else:

    print(
        "NOT TESTABLE: Dataset does not contain "
        "'feature_timestamp' and 'label_timestamp'."
    )

    print(
        "Temporal ordering cannot be verified from "
        "the available columns."
    )


# ============================================================
# 5. Final Summary
# ============================================================

print("\n" + "=" * 60)
print("LEAKAGE AUDIT SUMMARY")
print("=" * 60)

if suspect_features:
    print(
        "WARNING: Investigate the high-correlation features:"
    )
    print(suspect_features)
else:
    print(
        "PASS: No features exceed the 0.90 correlation threshold."
    )

print(
    "\nNote: Correlation alone does not prove leakage. "
    "Target-derived and future-window features must also "
    "be checked."
)

FEATURE-TARGET LEAKAGE CHECK

Feature correlations with target:
log_clicks                0.377475
log_impressions           0.332082
days_with_impressions     0.306063
days_with_sessions        0.247183
clicks_prev_30d           0.140435
impressions_prev_30d      0.139510
sessions_prev_30d         0.135681
impressions_90d           0.130606
sessions_90d              0.128000
users_90d                 0.127411
pageviews_90d             0.123126
clicks_90d                0.115192
engaged_sessions_90d      0.104696
avg_position              0.095203
scroll_rate               0.091648
click_drop_pct            0.086429
scroll_events_90d         0.086152
sessions_last_30d         0.085935
impressions_last_30d      0.074895
engagement_rate           0.058107
ai_sessions_90d           0.056988
days_since_last_update    0.048824
clicks_last_30d           0.045106
ctr                       0.038375
word_count                0.036316
char_count                0.030620
trend_pct                 

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Bold Original Claim: "Our machine learning model guarantees a 35% boost in ranking positions and predicts user conversion with 92% accuracy across all web properties."

Rewritten Safe Claim: "In offline evaluation on out-of-fold domain splits, the model demonstrated a directional positive correlation with rank improvements, achieving an observed 0.715 ROC-AUC score to serve as decision-support for content optimization strategies."

In [5]:
# ============================================================
# Claim Verification Summary
# ============================================================

print("--- Claim Verification Summary ---")

print(
    f"Evaluation Metric (ROC-AUC): {auc_group:.3f}"
)

print(
    "Validation Protocol: 5-Fold GroupKFold by Client ID"
)

print(
    "Grouping Variable: client_id"
)

print(
    "Claim Type: Directional / Decision-Support "
    "(Offline Validated)"
)

--- Claim Verification Summary ---
Evaluation Metric (ROC-AUC): 0.732
Validation Protocol: 5-Fold GroupKFold by Client ID
Grouping Variable: client_id
Claim Type: Directional / Decision-Support (Offline Validated)


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.